# Prepare data for IQ - HG

In [1]:
library(tidyverse)
library(tgutil)
library(misha)
library(misha.ext)
library(here)
library(prego)
library(iceqream)
gsetroot(here("data/hg19"))
options(gmax.data.size = 1e9)
doMC::registerDoMC(90)
theme_set(theme_classic())

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.1     ✔ tibble    3.2.1
✔ lubridate 1.9.3     ✔ tidyr     1.3.1
✔ purrr     1.0.2     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors
here() starts at /net/mraid20/ifs/wisdom/tanay_lab/tgdata/users/evghenic/Proj/polycomb/seq2epi_paper_norm



In [2]:
setwd(here())

In [3]:
cgd <- readr::read_rds(here("data/cgd_for_IQhg19.rds"))
tail(cgd)
dim(cgd)

,chrom,start,end,l,es_k27_max,es_k4_max,es_k27_cov,es_k4_cov,es_biv_cov,es_k27_cov_1,⋯,h1_k27_sum,l10_h1_k4_sum,l10_h1_k27_sum,pcg,txg,l500,f_out,txg_pcg,resp10,f_ambig
,<fct>,<dbl>,<dbl>,<dbl>,<dbl[1d]>,<dbl[1d]>,<dbl[1d]>,<dbl[1d]>,<dbl[1d]>,<dbl[1d]>,⋯,<dbl>,<dbl>,<dbl>,<lgl>,<lgl>,<lgl>,<lgl>,<dbl>,<dbl>,<lgl>
53526,chrX,154033252,154034117,865,23,38.10000,0,0.0000000,0,0,⋯,15.6069364,5.451640,4.678463,FALSE,FALSE,TRUE,TRUE,NA,NA,TRUE
53530,chrX,154254821,154255814,993,0,79.25000,0,0.6000000,0,0,⋯,0.0000000,6.303836,3.321928,FALSE,FALSE,TRUE,FALSE,0.6187212,NA,FALSE
53531,chrX,154299027,154300319,1292,1,108.55000,0,0.9846154,0,0,⋯,0.2321981,6.738657,3.355044,FALSE,FALSE,TRUE,FALSE,0.7032384,NA,FALSE
53532,chrX,154444440,154445336,896,3,94.64999,0,0.5777778,0,0,⋯,1.0044643,6.409107,3.460017,FALSE,FALSE,TRUE,FALSE,0.6167770,NA,FALSE
53534,chrX,154493243,154493936,693,0,71.55000,0,0.4000000,0,0,⋯,0.0000000,6.248256,3.321928,FALSE,FALSE,TRUE,FALSE,0.6071889,NA,FALSE
53544,chrX,154842089,154842785,696,0,92.34999,0,0.6666667,0,0,⋯,0.0000000,6.512093,3.321928,FALSE,FALSE,TRUE,FALSE,0.6619329,NA,FALSE


[1] 34153    43

In [5]:
test_chroms <- c("chr2", "chr8", "chr12", "chr18")

In [6]:
cgd <- cgd %>%
    gintervals.neighbors1("intervs.global.tss") %>%
    mutate(tss_gene = as.character(name)) %>%
    select(-(chrom1:dist))

In [8]:
cgd <- cgd %>%
    arrange(tss_gene, chrom, start, end) %>%
    group_by(tss_gene) %>%
    mutate(id = paste0(tss_gene, ".", row_number())) %>%
    mutate(n = n()) %>%
    mutate(id = ifelse(n == 1, tss_gene, id)) %>%
    select(-n) %>%
    ungroup() %>%
    select(chrom:end, id, everything()) 

In [9]:
fwrite(cgd, here("output/cgd_hg19_iq.tsv"), sep = "\t", quote = FALSE)

## Compute energies

In [8]:
mdb <- create_motif_db(iceqream::motif_db, prior = 0.01)

In [9]:
pwm <- gextract_pwm(cgd, dataset = mdb, bidirect = TRUE) %cache_rds% here("output/pwm_in_hg19_iq.rds")
cli::cli_alert_info("Finished extracting pwm for CGD")

ℹ Finished extracting pwm for CGD



In [10]:
head(pwm) %>% head()

,chrom,start,end,id,l,es_k27_max,es_k4_max,es_k27_cov,es_k4_cov,es_biv_cov,⋯,SCENIC.yetfasco__YPR008W_1425,SCENIC.yetfasco__YPR009W_2236,SCENIC.yetfasco__YPR013C_859,SCENIC.yetfasco__YPR052C_879,SCENIC.yetfasco__YPR054W_1875,SCENIC.yetfasco__YPR065W_1396,SCENIC.yetfasco__YPR086W_1327,SCENIC.yetfasco__YPR186C_1321,SCENIC.yetfasco__YPR196W_861,SCENIC.yetfasco__YPR199C_603
,<fct>,<dbl>,<dbl>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,chr1,9488701,9489371,5S_rRNA.1,670,5.00,25.60,0.00000000,0.0000000,0.00000000,⋯,-0.67710167,-6.518007,-23.16786,-22.48188,-19.35524,-5.942288,-5.941751,-22.75656,-22.75592,-3.750360
2,chr1,41826605,41828380,5S_rRNA.2,1775,110.25,90.60,0.38202247,0.0000000,0.33707865,⋯,1.20072198,-6.262984,-24.70769,-28.80173,-22.58040,-3.653504,-6.489796,-22.74199,-25.65578,-6.337624
3,chr1,41831862,41832691,5S_rRNA.3,829,17.00,88.75,0.00000000,0.7619048,0.00000000,⋯,1.08987427,-7.471943,-20.99963,-23.40126,-23.67208,-6.929895,-5.592474,-21.84194,-24.28015,-5.177572
4,chr1,41847183,41849391,5S_rRNA.4,2208,29.45,117.05,0.05405405,0.8018018,0.01801802,⋯,0.96127445,-6.441759,-24.20991,-28.65687,-22.91767,-7.157766,-7.493325,-22.35264,-25.06879,-5.653064
5,chr1,41889237,41889828,5S_rRNA.5,591,0.00,2.00,0.00000000,0.0000000,0.00000000,⋯,-0.01841912,-6.603125,-24.22689,-26.30877,-15.68951,-5.188820,-5.901658,-18.86879,-24.64464,-5.951372
6,chr1,117487141,117487793,5S_rRNA.6,652,2.00,0.00,0.00000000,0.0000000,0.00000000,⋯,0.08850773,-6.430099,-22.67188,-23.15333,-23.57291,-5.218352,-7.763625,-22.71868,-22.14412,-5.290758


In [ ]:
mat_in <- pwm %>%
    select(-any_of(setdiff(colnames(cgd), "id"))) %>%
    remove_rownames() %>%
    column_to_rownames("id") %>%
    as.matrix()
mat_in <- mat_in[rowSums(!is.na(mat_in)) > 0, ] 
pwm_norm <- norm_energy_matrix(mat_in, mat_in, min_energy = -12, q = 0.995) %cache_rds% here("output/pwm_in_norm_hg19_iq.rds")

In [12]:
cgd5 <- cgd[, 1:4]
cgd5$start <- ifelse(cgd$tss_strand == 1, cgd$start, cgd$end - 600)
cgd5$end <- ifelse(cgd$tss_strand == 1, cgd$start + 600, cgd$end)

cgd5_pad <- cgd[, 1:4]
cgd5_pad$start <- ifelse(cgd$tss_strand == 1, cgd$start - 500, cgd$end)
cgd5_pad$end <- ifelse(cgd$tss_strand == 1, cgd$start, cgd$end + 500)

cgd3 <- cgd[, 1:4]
cgd3$start <- ifelse(cgd$tss_strand == 1, cgd$end - 600, cgd$start)
cgd3$end <- ifelse(cgd$tss_strand == 1, cgd$end, cgd$start + 600)

cgd3_pad <- cgd[, 1:4]
cgd3_pad$start <- ifelse(cgd$tss_strand == 1, cgd$end, cgd$start - 500)
cgd3_pad$end <- ifelse(cgd$tss_strand == 1, cgd$end + 500, cgd$start)

In [ ]:
pwm5 <- gextract_pwm(cgd5, dataset = mdb, bidirect = TRUE) %cache_rds% here("output/pwm5_hg19_iq.rds")
cli::cli_alert_info("Finished extracting pwm for CGD5")
pwm5_pad <- gextract_pwm(cgd5_pad, dataset = mdb, bidirect = TRUE) %cache_rds% here("output/pwm5_pad_hg19_iq.rds")
cli::cli_alert_info("Finished extracting pwm for CGD5_PAD")
pwm3 <- gextract_pwm(cgd3, dataset = mdb, bidirect = TRUE) %cache_rds% here("output/pwm3_hg19_iq.rds")
cli::cli_alert_info("Finished extracting pwm for CGD3")
pwm3_pad <- gextract_pwm(cgd3_pad, dataset = mdb, bidirect = TRUE) %cache_rds% here("output/pwm3_pad_hg19_iq.rds")
cli::cli_alert_info("Finished extracting pwm for CGD3_PAD")

ℹ Finished extracting pwm for CGD5

ℹ Finished extracting pwm for CGD5_PAD

ℹ Finished extracting pwm for CGD3

ℹ Finished extracting pwm for CGD3_PAD



In [ ]:
mat5 <- pwm5 %>%
    select(-(chrom:end)) %>%
    column_to_rownames("id") %>%
    as.matrix()
mat5 <- mat5[rowSums(!is.na(mat5)) > 0, ]
pwm5_norm <- norm_energy_matrix(mat5, mat5, min_energy = -12, q = 0.995) %cache_rds% here("output/pwm5_norm_hg19_iq.rds")

In [15]:
mat5_pad <- pwm5_pad %>%
    select(-(chrom:end)) %>%
    column_to_rownames("id") %>%
    as.matrix()
mat5_pad <- mat5_pad[rowSums(!is.na(mat5_pad)) > 0, ] 
pwm5_pad_norm <- norm_energy_matrix(mat5_pad, mat5_pad, min_energy = -12, q = 0.995) %cache_rds% here("output/pwm5_pad_norm_hg19_iq.rds")
mat3 <- pwm3 %>%
    select(-(chrom:end)) %>%
    column_to_rownames("id") %>%
    as.matrix()
mat3 <- mat3[rowSums(!is.na(mat3)) > 0, ] 
pwm3_norm <- norm_energy_matrix(mat3, mat3, min_energy = -12, q = 0.995) %cache_rds% here("output/pwm3_norm_hg19_iq.rds")
mat3_pad <- pwm3_pad %>%
    select(-(chrom:end)) %>%
    column_to_rownames("id") %>%
    as.matrix()
mat3_pad <- mat3_pad[rowSums(!is.na(mat3_pad)) > 0, ]
pwm3_pad_norm <- norm_energy_matrix(mat3_pad, mat3_pad, min_energy = -12, q = 0.995) %cache_rds% here("output/pwm3_pad_norm_hg19_iq.rds")

## Create additional features

In [16]:
din <- create_sequence_features(cgd)
rownames(din) <- cgd$id
readr::write_rds(din, here("output/add_feats_in_hg19_iq.rds"))

In [17]:
din5 <- create_sequence_features(cgd5)
colnames(din5) <- paste0(colnames(din5), "_5")
din5_pad <- create_sequence_features(cgd5_pad)
colnames(din5_pad) <- paste0(colnames(din5_pad), "_5pad")
din3 <- create_sequence_features(cgd3)
colnames(din3) <- paste0(colnames(din3), "_3")
din3_pad <- create_sequence_features(cgd3_pad)
colnames(din3_pad) <- paste0(colnames(din3_pad), "_3pad")
add_feats <- cbind(
    din5,
    din5_pad,
    din3,
    din3_pad
)
rownames(add_feats) <- cgd$id
readr::write_rds(add_feats, here("output/add_feats_hg19_iq.rds"))